In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType

catalog_name = 'ecommerce'

In [0]:
# 0.1 Initialization: loading bronze into silver data frame
df_silver_date = spark.table(f"{catalog_name}.bronze.brz_date")

df_silver_date.printSchema()

In [0]:
# 0.2 Initialization: Printing number of rows & columns

row_count, column_count = df_silver_date.count(), len(df_silver_date.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")


In [0]:
# 0.3 Initialization: Standardizing dates as PK

df_silver_date = df_silver_date.withColumnRenamed(
    "date", "date_id"
)

df_silver_date = df_silver_date.withColumn(
    "date_id",
    F.regexp_replace(F.col("date_id"), "-", "/")
)

df_silver_date = df_silver_date.withColumn(
    "date_id",
    F.coalesce(
        F.expr("try_to_date(date_id, 'M/d/yy')"),
        F.expr("try_to_date(date_id, 'M/dd/yy')"),
        F.expr("try_to_date(date_id, 'M/d/yyyy')"),
        F.expr("try_to_date(date_id, 'M/dd/yyyy')"),

        F.expr("try_to_date(date_id, 'MM/d/yy')"),
        F.expr("try_to_date(date_id, 'MM/dd/yy')"),
        F.expr("try_to_date(date_id, 'MM/d/yyyy')"),
        F.expr("try_to_date(date_id, 'MM/dd/yyyy')"),

        F.expr("try_to_date(date_id, 'd/M/yy')"),
        F.expr("try_to_date(date_id, 'd/MM/yy')"),
        F.expr("try_to_date(date_id, 'd/M/yyyy')"),
        F.expr("try_to_date(date_id, 'd/MM/yyyy')"),

        F.expr("try_to_date(date_id, 'dd/M/yy')"),
        F.expr("try_to_date(date_id, 'dd/MM/yy')"),
        F.expr("try_to_date(date_id, 'dd/M/yyyy')"),
        F.expr("try_to_date(date_id, 'dd/MM/yyyy')"),

        F.expr("try_to_date(date_id, 'yy/M/d')"),
        F.expr("try_to_date(date_id, 'yy/MM/d')"),
        F.expr("try_to_date(date_id, 'yy/M/dd')"),
        F.expr("try_to_date(date_id, 'yy/MM/dd')"),

        F.expr("try_to_date(date_id, 'yyyy/M/d')"),
        F.expr("try_to_date(date_id, 'yyyy/MM/d')"),
        F.expr("try_to_date(date_id, 'yyyy/M/dd')"),
        F.expr("try_to_date(date_id, 'yyyy/MM/dd')")
    )
)

from pyspark.sql import Window

df_silver_date = df_silver_date.withColumn(
    "derived_quarter", F.quarter(F.col("date_id"))
)

df_silver_date = df_silver_date.withColumn(
    "date_id",
    F.when(
        F.col("derived_quarter") != F.col("quarter"),

        F.to_date(
            F.concat_ws("/",
                F.year(F.col("date_id")).cast("string"),
                F.dayofmonth(F.col("date_id")).cast("string"),
                F.month(F.col("date_id")).cast("string"),
                
                ), "yyyy/M/d"
        )
    ).otherwise(F.col("date_id"))
)

df_silver_date = df_silver_date.orderBy("date_id")

In [0]:
df_silver_date.withColumn(
    "derived_quarter", F.quarter(F.col("date_id"))
).filter(
    F.col("derived_quarter") != F.col("quarter")
).select("date_id", "quarter", "derived_quarter").show()

df_silver_date.printSchema()

In [0]:
# 0.4 Initialization: Checking for duplicates

total_rows = df_silver_date.count()
distinct_rows = df_silver_date.select("date_id").distinct().count()
print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicates: {total_rows - distinct_rows}")

fully_distinct_rows = df_silver_date.distinct().count()

print(f"Fully distinct rows: {fully_distinct_rows}")
print(f"Distinct rows: {distinct_rows}")

## 1. date

In [0]:
# 1.1 transformation: slv_date -> date

df_silver_date = df_silver_date.dropDuplicates(["date_id"])


In [0]:
# 1.2 validation: slv_date -> date
from datetime import date

today = date.today()

df_silver_date.select("date_id").filter(
    F.col("date_id").isNull() | (F.col("date_id") > F.lit(today)) | (F.col("date_id") < F.lit(date(1900, 1, 1)))
).show()

df_silver_date.groupBy("date_id").count() \
    .filter(F.col("count") > 1).show()

## 2. year

In [0]:
# 2.1 transformation: slv_date -> year



In [0]:
# 2.2 validation: slv_date -> year

df_silver_date.select("year").filter(
    F.col("year").isNull() | (F.col("year") < 1900) | (F.col("year") > today.year)
    ).show()

## 3. day_name

In [0]:
# 3.1 transformation: slv_date -> day_name

df_silver_date = df_silver_date.withColumn(
  "day_name", F.initcap(F.trim(F.col("day_name")))
)

In [0]:
# 3.2 validation: slv_date -> day_name

valid_day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

df_silver_date.select("day_name").filter(
  ~F.col("day_name").isin(valid_day_names)
).show()

## 4. quarter

In [0]:
# 4.1 transformation: slv_date -> quarter




In [0]:
# 4.2 validation: slv_date -> quarter

df_silver_date.select("quarter").filter(
    (F.col("quarter") < 1) | (F.col("quarter") > 4) | (F.col("quarter").isNull())
).show()

## 5. week_of_year

In [0]:
# 5.1 transformation: slv_date -> week_of_year
df_silver_date = df_silver_date.withColumn(
    "week_of_year",
    F.abs(F.col("week_of_year"))
)

In [0]:
# 5.2 validation: slv_date -> week_of_year

df_silver_date.select("week_of_year").filter(
    (F.col("week_of_year") < 1) | (F.col("week_of_year") > 53) | (F.col("week_of_year").isNull())
).show()

## Quarantine vs Clean Data

In [0]:
# 6.1 Quarantine Bad Data -> df_silver_date_clean & df_silver_date_quarantine

df_silver_date_clean = df_silver_date.filter(
  F.col("date_id").isNotNull() & (F.col("date_id") > "1900-01-01") & (F.col("date_id") < today) &
  F.col("year").isNotNull() & (F.col("year") > 1900) & (F.col("year") < 2100) &
  F.col("day_name").isNotNull() & (F.col("day_name") != "") & (F.col("day_name").isin(valid_day_names)) &
  F.col("quarter").isNotNull() & (F.col("quarter") > 0) & (F.col("quarter") < 5) &
  F.col("week_of_year").isNotNull() & (F.col("week_of_year") > 0) & (F.col("week_of_year") < 54)

)

df_silver_date_quarantine = df_silver_date.filter(
  F.col("date_id").isNull() | (F.col("date_id") < "1900-01-01") | (F.col("date_id") > today) |
  F.col("year").isNull() | (F.col("year") < 1900) | (F.col("year") > 2100) |
  F.col("day_name").isNull() | (F.col("day_name") == "") | (~F.col("day_name").isin(valid_day_names)) |
  F.col("quarter").isNull() | (F.col("quarter") < 1) | (F.col("quarter") > 4) |
  F.col("week_of_year").isNull() | (F.col("week_of_year") < 1) | (F.col("week_of_year") > 53)) \
    .withColumn("rejection_reason",
      F.when(F.col("date_id").isNull() | (F.col("date_id") < "1900-01-01") | (F.col("date_id") > today), "null/invalid date_id")
      .when(F.col("year").isNull() | (F.col("year") < 1900) | (F.col("year") > 2100), "null/invalid year")
      .when(F.col("day_name").isNull() | (F.col("day_name") == "") | (~F.col("day_name").isin(valid_day_names)), "null/empty/invalid day_name")
      .when(F.col("quarter").isNull() | (F.col("quarter") < 1) | (F.col("quarter") > 4), "null/invalid quarter")
      .when(F.col("week_of_year").isNull() | (F.col("week_of_year") < 1) | (F.col("week_of_year") > 53), "null/invalid week_of_year")
    )

In [0]:
# 6.2 Check Quarantined Data
df_silver_date_quarantine.show()


In [0]:
# 6.3 Write to Delta Tables -> df_silver_date_clean & df_silver_date_quarantine

df_silver_date_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_date_clean")

df_silver_date_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_date_quarantine")
